# Session 2 — Hand Tracking & Gesture Recognition
### Applied Computer Vision with MediaPipe

## Pipeline we will build

```
Image / Webcam
      ↓
Hand Detection
      ↓
Hand Landmarks (21 points)
      ↓
Features
      ↓
Finger States
      ↓
Gesture Recognition
      ↓
Action
```

**Goal:** turn raw model output into meaningful information.

Final target mapping:

| Gesture     | Action    |
|-------------|-----------|
| Open Hand   | START     |
| Fist        | STOP      |
| Pointing    | NEXT      |
| Peace       | PREVIOUS  |

We build this piece by piece.

## 1. Quick Review of Session 1

1. What is a landmark?
2. What do `x` and `y` represent?
3. Why are MediaPipe coordinates normalized?
4. Difference between an image and a video frame?

Reconnect the tools from Session 1.

In [1]:
#!pip install mediapipe opencv-python matplotlib numpy pillow -q

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import cv2
import numpy as np
import matplotlib.pyplot as plt
import urllib.request
import os
import math

print("MediaPipe:", mp.__version__)

MediaPipe: 1.0.1


In [2]:
MODEL_URL = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
MODEL_PATH = "hand_landmarker.task"

if not os.path.exists(MODEL_PATH):
    print("Downloading model...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
print("Model ready.")

Model ready.


In [ ]:
IMAGE_URL = "https://storage.googleapis.com/mediapipe-tasks/hand_landmarker/woman_hands.jpg"
IMAGE_PATH = "sample_hands.jpg"

if not os.path.exists(IMAGE_PATH):
    print("Downloading sample image...")
    urllib.request.urlretrieve(IMAGE_URL, IMAGE_PATH)
print("Image ready.")

In [ ]:
BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.IMAGE,
    num_hands=2
)
landmarker = HandLandmarker.create_from_options(options)
print("Hand Landmarker ready.")

In [ ]:
mp_image = mp.Image.create_from_file(IMAGE_PATH)
result = landmarker.detect(mp_image)

print(f"Hands detected: {len(result.hand_landmarks)}")
if result.hand_landmarks:
    print(f"Landmarks per hand: {len(result.hand_landmarks[0])}")

**Checkpoint:** You should see at least one hand with 21 landmarks.

## 2. Understanding the 21 Hand Landmarks

MediaPipe returns 21 landmarks per hand.  
Each is a normalized 3-D point `(x, y, z)`.

![MediaPipe 21 Hand Landmarks](https://mediapipe.dev/images/mobile/hand_landmarks.png)

*Source: MediaPipe documentation*

| Index | Name              |
|-------|-------------------|
| 0     | WRIST             |
| 4     | THUMB_TIP         |
| 8     | INDEX_FINGER_TIP  |
| 12    | MIDDLE_FINGER_TIP |
| 16    | RING_FINGER_TIP   |
| 20    | PINKY_TIP         |

Intermediate joints (MCP, PIP, DIP) are used later for finger-state rules.

In [ ]:
LANDMARK_NAMES = {
    0: "WRIST",
    1: "THUMB_CMC", 2: "THUMB_MCP", 3: "THUMB_IP", 4: "THUMB_TIP",
    5: "INDEX_MCP", 6: "INDEX_PIP", 7: "INDEX_DIP", 8: "INDEX_TIP",
    9: "MIDDLE_MCP", 10: "MIDDLE_PIP", 11: "MIDDLE_DIP", 12: "MIDDLE_TIP",
    13: "RING_MCP", 14: "RING_PIP", 15: "RING_DIP", 16: "RING_TIP",
    17: "PINKY_MCP", 18: "PINKY_PIP", 19: "PINKY_DIP", 20: "PINKY_TIP",
}

from mediapipe.tasks.python.vision import HandLandmarksConnections
HAND_CONNECTIONS = HandLandmarksConnections.HAND_CONNECTIONS
print("Connections loaded:", len(HAND_CONNECTIONS))

## 3. Inspecting Landmark Data

The model returns numbers, not gesture labels.

In [ ]:
a = result.hand_landmarks

if not result.hand_landmarks:
    raise ValueError("No hands detected. Try another image.")

hand = result.hand_landmarks[0]

for idx in [0, 4, 8, 12]:
    lm = hand[idx]
    print(f"{idx:2d} ({LANDMARK_NAMES[idx]:12s}): "
          f"x={lm.x:.4f}  y={lm.y:.4f}  z={lm.z:.4f}")

- `x`, `y` are normalized to [0, 1]
- `z` is relative depth (smaller = closer to camera)

In [ ]:
img_bgr = cv2.imread(IMAGE_PATH)
h, w = img_bgr.shape[:2]
print(f"Image size: {w} x {h}")

def to_pixel(lm, width, height):
    return int(lm.x * width), int(lm.y * height)

for idx in [0, 8, 12]:
    px, py = to_pixel(hand[idx], w, h)
    print(f"{LANDMARK_NAMES[idx]:12s} -> pixel ({px}, {py})")

**Checkpoint:** We have 21 numerical points. What can we calculate from them?

## 4. Visualizing the Hand

Build the skeleton step by step:

```
Raw Image  ->  21 Points  ->  Skeleton
```

In [ ]:
def draw_point(img, lm, color=(0, 255, 0), radius=6):
    h, w = img.shape[:2]
    x, y = int(lm.x * w), int(lm.y * h)
    cv2.circle(img, (x, y), radius, color, -1)
    return img

# Step 1 — selected landmarks only
vis = img_bgr.copy()
for idx in hand:
    draw_point(vis, idx, color=(255, 0, 0), radius=3)

plt.figure(figsize=(8, 10))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
#plt.imshow(vis)
plt.title("Step 1 — Wrist + fingertips")
plt.axis("off")
plt.show()

In [ ]:
def draw_connections(img, landmarks, connections, color=(255, 255, 255), thickness=2):
    h, w = img.shape[:2]
    for connection in connections:
        start = landmarks[connection.start]
        end   = landmarks[connection.end]
        p1 = (int(start.x * w), int(start.y * h))
        p2 = (int(end.x * w),   int(end.y * h))
        cv2.line(img, p1, p2, color, thickness)
    return img

# Step 3 — full skeleton
vis = img_bgr.copy()
draw_connections(vis, hand, HAND_CONNECTIONS, color=(0, 255, 100), thickness=2)
for lm in hand:
    draw_point(vis, lm, color=(0, 100, 255), radius=4)

plt.figure(figsize=(8, 10))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title("Step 3 — Hand skeleton")
plt.axis("off")
plt.show()

**Checkpoint:** The 21 numbers are now a visible hand skeleton.

## 5. Is a Landmark a Gesture?

```
21 landmarks
      ↓
      ?
      ↓
"Peace Sign"
```

No. Landmarks are raw geometry.  
We must extract **relationships** between them.

```
Landmarks  ->  Features  ->  Decisions
```

## 6. Feature Extraction

Useful features: distance, relative position, vertical/horizontal relations.

In [ ]:
def distance(p1, p2):
    """Euclidean distance between two landmarks."""
    return math.sqrt((p1.x - p2.x)**2 + (p1.y - p2.y)**2 + (p1.z - p2.z)**2)

wrist     = hand[0]
index_tip = hand[8]
thumb_tip = hand[4]
index_mcp = hand[5]

print(f"Wrist -> Index Tip : {distance(wrist, index_tip):.4f}")
print(f"Thumb Tip -> Index Tip : {distance(thumb_tip, index_tip):.4f}")
print(f"Index MCP -> Index Tip : {distance(index_mcp, index_tip):.4f}")

In [ ]:
def draw_line_and_label(img, lm1, lm2, label, color=(0, 255, 255)):
    h, w = img.shape[:2]
    p1 = (int(lm1.x * w), int(lm1.y * h))
    p2 = (int(lm2.x * w), int(lm2.y * h))
    cv2.line(img, p1, p2, color, 2)
    mid = ((p1[0]+p2[0])//2, (p1[1]+p2[1])//2)
    cv2.putText(img, label, mid, cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

vis = img_bgr.copy()
draw_line_and_label(vis, wrist, index_tip, f"{distance(wrist, index_tip):.3f}")
draw_point(vis, wrist, (255, 0, 0))
draw_point(vis, index_tip, (0, 255, 0))

plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title("Feature: Wrist -> Index Tip distance")
plt.axis("off")
plt.show()

Relationships are more useful than absolute coordinates because they are more invariant to hand position in the frame.

## 7. Finger State Detection

Start with one finger (Index):

```
MCP (5) -- PIP (6) -- DIP (7) -- TIP (8)
```

Simple rule: finger is **open** if tip is farther from the wrist than the PIP joint.

In [ ]:
def is_finger_open(landmarks, tip_idx, pip_idx, wrist_idx=0):
    """Open if tip is farther from wrist than PIP."""
    tip   = landmarks[tip_idx]
    pip   = landmarks[pip_idx]
    wrist = landmarks[wrist_idx]
    return distance(tip, wrist) > distance(pip, wrist)

index_open = is_finger_open(hand, tip_idx=8, pip_idx=6)
print("Index open?", index_open)

In [ ]:
vis = img_bgr.copy()
draw_line_and_label(vis, hand[0], hand[8], "tip",
                    color=(0, 255, 0) if index_open else (0, 0, 255))
draw_line_and_label(vis, hand[0], hand[6], "pip", color=(255, 200, 0))

status = "OPEN" if index_open else "CLOSED"
cv2.putText(vis, f"Index: {status}", (30, 40),
            cv2.FONT_HERSHEY_SIMPLEX, 1.0,
            (0, 255, 0) if index_open else (0, 0, 255), 2)

plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title("Index finger state")
plt.axis("off")
plt.show()

This rule works best for frontal, upright hands.  
Thumb needs a slightly different treatment.

In [ ]:
def finger_states(landmarks):
    """Return [thumb, index, middle, ring, pinky] as booleans."""
    tips = [4, 8, 12, 16, 20]
    pips = [3, 6, 10, 14, 18]
    return [is_finger_open(landmarks, t, p) for t, p in zip(tips, pips)]

states = finger_states(hand)
names = ["Thumb", "Index", "Middle", "Ring", "Pinky"]
print("Finger states:")
for n, s in zip(names, states):
    print(f"  {n:6s}: {'OPEN' if s else 'CLOSED'}")

**Checkpoint:** We now have a compact hand representation  
`[thumb, index, middle, ring, pinky]`.

## 8. From Finger States to Gestures

```
Finger States  ->  Pattern  ->  Gesture
```

In [ ]:
def recognize_gesture(states):
    """Simple rule-based recognizer."""
    thumb, index, middle, ring, pinky = states

    if index and middle and ring and pinky and thumb:
        return "Open Hand"
    if not index and not middle and not ring and not pinky:
        return "Fist"
    if index and not middle and not ring and not pinky:
        return "Pointing"
    if index and middle and not ring and not pinky:
        return "Peace"
    return "Unknown"

gesture = recognize_gesture(states)
print("Gesture:", gesture)

In [ ]:
vis = img_bgr.copy()
draw_connections(vis, hand, HAND_CONNECTIONS, color=(120, 120, 120), thickness=2)
for lm in hand:
    draw_point(vis, lm, color=(0, 180, 255), radius=4)

cv2.putText(vis, gesture, (30, 50),
            cv2.FONT_HERSHEY_SIMPLEX, 1.4, (0, 255, 100), 3)

plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title(f"Gesture: {gesture}")
plt.axis("off")
plt.show()

## 9. Rule-Based Recognition — Pros and Cons

**Advantages**
- Easy to understand and debug
- Fast, no training data needed
- Good for a small set of clear gestures

**Limitations**
- Sensitive to orientation and viewpoint
- Fails with occlusion or extreme poses
- Hard to scale to many complex gestures

This is different from a trained ML gesture classifier.

## 10. Multiple Hands

`result.hand_landmarks` is a list. Loop over it.

In [ ]:
print(f"Hands: {len(result.hand_landmarks)}")

for i, (landmarks, handedness) in enumerate(zip(result.hand_landmarks, result.handedness)):
    label = handedness[0].category_name
    score = handedness[0].score
    gest = recognize_gesture(finger_states(landmarks))
    print(f"Hand {i}: {label} ({score:.2f}) -> {gest}")

Always handle 0, 1, or 2 hands. Never assume index 0 exists.

## 11. From Image to Webcam

```
Webcam -> Frame -> MediaPipe -> Landmarks -> Gesture
```

Every frame is processed independently. FPS determines smoothness.

## 12. Real-Time Hand Tracking

Build helpers first, then the loop. Press `q` to quit.

In [ ]:
def get_gesture(landmarks):
    return recognize_gesture(finger_states(landmarks))

def draw_hand(img, landmarks, gesture=None):
    """Draw skeleton + optional gesture label."""
    draw_connections(img, landmarks, HAND_CONNECTIONS, color=(0, 255, 100), thickness=2)
    for lm in landmarks:
        draw_point(img, lm, color=(0, 150, 255), radius=4)
    if gesture:
        cv2.putText(img, gesture, (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 2)
    return img

In [ ]:
options_video = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_tracking_confidence=0.5
)
landmarker_video = HandLandmarker.create_from_options(options_video)
print("Video-mode landmarker ready.")

In [ ]:
# Real-time demo (press 'q' to quit)
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Could not open webcam. Skipping live demo.")
else:
    print("Webcam opened. Press 'q' to quit.")
    frame_timestamp_ms = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.flip(frame, 1)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_frame = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

        result_frame = landmarker_video.detect_for_video(mp_frame, frame_timestamp_ms)
        frame_timestamp_ms += 33

        for landmarks in result_frame.hand_landmarks:
            gest = get_gesture(landmarks)
            draw_hand(frame, landmarks, gest)

        cv2.imshow("Real-time Hand Tracking", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    print("Webcam closed.")

## 13. Gestures Trigger Actions

```
Gesture  ->  Action
```

In [ ]:
ACTION_MAP = {
    "Open Hand": "START",
    "Fist":      "STOP",
    "Pointing":  "NEXT",
    "Peace":     "PREVIOUS",
}

def gesture_to_action(gesture):
    return ACTION_MAP.get(gesture, None)

print(f"'{gesture}' -> '{gesture_to_action(gesture)}'")

Later the action can become a keyboard event, API call, robot command, slide change, etc.

## 14. Practical Examples

**Presentation controller**  
Pointing = next slide, Peace = previous, Open Hand = pause.

**Media controller**  
Open Hand = play, Fist = pause. (Add thumb-up / thumb-down for volume.)

**Accessibility**  
Gestures as alternative input when mouse/keyboard are difficult.

**HCI / Installations**  
Touchless kiosks, smart displays, AR/VR, interactive exhibits.

**Robotics (conceptual)**  
Gesture -> CV pipeline -> robot command (stop, move, grab).

## 15. Problem: Repeated Actions

Holding "Pointing" for 2 seconds would fire NEXT many times.

Solution — trigger only on change:

```
previous = None
if current != previous:
    trigger(current)
    previous = current
```

In [ ]:
class GestureActionFilter:
    def __init__(self):
        self.previous = None

    def update(self, gesture):
        """Return action only when gesture changes."""
        if gesture != self.previous and gesture != "Unknown":
            action = gesture_to_action(gesture)
            self.previous = gesture
            return action
        self.previous = gesture
        return None

filt = GestureActionFilter()
for g in ["Unknown", "Pointing", "Pointing", "Pointing", "Peace", "Peace", "Fist"]:
    print(f"{g:10s} -> {filt.update(g)}")

## 16. Mini Project — Virtual Gesture Controller

Assemble the pieces already built:

```
Webcam -> Frame -> MediaPipe -> Landmarks
       -> Features -> Finger States -> Gesture
       -> Action Mapping -> Feedback
```

In [ ]:
import time

def run_gesture_controller(max_frames=300):
    """Full pipeline. Press 'q' to quit. max_frames limits notebook runtime."""
    # New landmarker every run -> no leftover timestamps
    options_live = HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=MODEL_PATH),
        running_mode=VisionRunningMode.VIDEO,
        num_hands=2,
        min_hand_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    landmarker_live = HandLandmarker.create_from_options(options_live)

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Webcam not available.")
        landmarker_live.close()
        return

    action_filter = GestureActionFilter()
    frame_count = 0
    last_ts = 0

    print("Controller running. Press 'q' to quit.")
    try:
        while frame_count < max_frames:
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.flip(frame, 1)
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_frame = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

            # Absolute wall-clock time in ms
            timestamp_ms = int(time.time() * 1000)
            if timestamp_ms <= last_ts:
                timestamp_ms = last_ts + 1
            last_ts = timestamp_ms

            result_frame = landmarker_live.detect_for_video(mp_frame, timestamp_ms)
            frame_count += 1

            current_gesture = "None"
            for landmarks in result_frame.hand_landmarks:
                current_gesture = get_gesture(landmarks)
                draw_hand(frame, landmarks, current_gesture)

            action = action_filter.update(current_gesture)
            if action:
                print(f">>> ACTION: {action}")
                cv2.putText(frame, f"ACTION: {action}", (20, 90),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)

            cv2.imshow("Virtual Gesture Controller", frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    finally:
        cap.release()
        cv2.destroyAllWindows()
        landmarker_live.close()
        print("Controller stopped.")

run_gesture_controller()

## 17. Code Cleanup (looking ahead)

In a real project the logic would be split, for example:

```
gesture_controller/
├── main.py
├── hand_detector.py
├── gesture_recognizer.py
└── utils.py
```

This notebook keeps everything sequential for teaching.

## 18. Debugging & Failure Cases

| Situation              | Typical effect                    |
|------------------------|-----------------------------------|
| Poor lighting          | Unstable or failed detection      |
| Occlusion              | Missing / wrong landmarks         |
| Extreme rotation       | Finger-state rules break          |
| Very far / very close  | Scale and confidence issues       |
| Multiple overlapping hands | Ambiguity                     |
| Low FPS                | Laggy interface                   |

Visualize intermediate results. Start in good lighting with a frontal hand.

## 19. Challenges

**Beginner**  
Add a Thumb-up gesture that maps to "CONFIRM".  
(Thumb open and roughly upward; other fingers closed.)

**Intermediate**  
Require a sequence before acting, e.g. Open Hand then Fist = "EMERGENCY STOP".  
Keep a short history of recent gestures.

**Advanced**  
Design a complete application concept (presentation controller, media player, simple game).  
Focus on the CV pipeline and clear action mapping.

## Closing

You started with 21 raw points and step by step turned them into:

```
features -> finger states -> gestures -> actions -> working application
```

**Takeaway:** A pre-trained model gives raw information.  
Applications are built by turning that information into features, decisions, and actions.